# SPY vs Fed Funds Correlation

This notebook:
- pulls SPY prices from Yahoo Finance
- loads local `FEDFUNDS.csv`
- engineers monthly features
- computes static and rolling correlations

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px

START_DATE = "2008-01-01"

pd.set_option("display.max_columns", 200)

In [ ]:
# Pull SPY (daily) and convert to month-end close + monthly log return
spy = yf.download("SPY", period="max", auto_adjust=False, progress=False)

if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)

spy = spy.reset_index().rename(columns={"Date": "date", "Datetime": "date"})
spy["date"] = pd.to_datetime(spy["date"])
spy = spy[spy["date"] >= pd.Timestamp(START_DATE)]

spy_m = (
    spy.set_index("date")[["Close"]]
    .resample("ME")
    .last()
    .rename(columns={"Close": "spy_close"})
)
spy_m["spy_log_ret_1m"] = np.log(spy_m["spy_close"]).diff()
spy_m.tail()

In [ ]:
# Load Fed Funds (monthly)
fed = pd.read_csv("../data/raw/FEDFUNDS.csv", parse_dates=["observation_date"])
fed = fed.rename(columns={"observation_date": "date", "FEDFUNDS": "fed_funds"})
fed = fed[fed["date"] >= pd.Timestamp(START_DATE)]
fed = fed.sort_values("date").set_index("date")

# Align to month-end index and forward-fill within month cadence
fed_m = fed.resample("ME").last().ffill()
fed_m["fed_delta_1m"] = fed_m["fed_funds"].diff()
fed_m["fed_delta_3m"] = fed_m["fed_funds"].diff(3)
fed_m.tail()

In [ ]:
# Join and engineer lagged features
df = spy_m.join(fed_m, how="inner")
for lag in [0, 1, 3, 6]:
    df[f"fed_delta_1m_lag_{lag}"] = df["fed_delta_1m"].shift(lag)

df = df.dropna(subset=["spy_log_ret_1m"])
df.tail()

In [ ]:
# Static correlations
targets = [
    "fed_funds",
    "fed_delta_1m",
    "fed_delta_3m",
    "fed_delta_1m_lag_1",
    "fed_delta_1m_lag_3",
    "fed_delta_1m_lag_6",
]

corr_rows = []
for col in targets:
    s = df[["spy_log_ret_1m", col]].dropna()
    corr_rows.append(
        {
            "feature": col,
            "pearson": s["spy_log_ret_1m"].corr(s[col], method="pearson"),
            "spearman": s["spy_log_ret_1m"].corr(s[col], method="spearman"),
            "n_obs": len(s),
        }
    )

corr_table = pd.DataFrame(corr_rows).sort_values("pearson", key=np.abs, ascending=False)
corr_table

In [ ]:
# Scatter plot: SPY log return vs Fed monthly change
scatter_df = df[["spy_log_ret_1m", "fed_delta_1m"]].dropna().reset_index()

fig = px.scatter(
    scatter_df,
    x="fed_delta_1m",
    y="spy_log_ret_1m",
    trendline="ols",
    title="SPY Monthly Log Return vs Fed Funds Monthly Change (since 2008)",
)
fig.update_layout(xaxis_title="Fed Funds Delta 1M", yaxis_title="SPY Log Return 1M")
fig.show()

In [ ]:
# Rolling 36-month correlation: SPY log return vs Fed monthly change
roll_window = 36
rolling = (
    df[["spy_log_ret_1m", "fed_delta_1m"]]
    .dropna()
    .assign(rolling_corr=lambda x: x["spy_log_ret_1m"].rolling(roll_window).corr(x["fed_delta_1m"]))
)

fig = px.line(
    rolling.reset_index(),
    x="date",
    y="rolling_corr",
    title=f"Rolling {roll_window}M Corr: SPY log return vs Fed delta 1M",
)
fig.add_hline(y=0, line_dash="dash")
fig.show()